# 原始影像下载与文件完整性

HTTP Range选择性提取官方ZIP中的原始DICOM，验证每个文件的ZIP CRC32、字节数和SHA256。
这些检查证明文件传输完整；不会替代配准、病灶可见性、伪影、序列身份及分割质量评估。


In [1]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/rn_public').is_dir())
sys.path.insert(0, str(ROOT / 'src'))
from rn_public.bundle import restore_sources
print('Source restoration:', restore_sources(ROOT))
print('Project root:', ROOT.name)


Source restoration: {'restored_files': 0, 'status': 'all_source_checksums_verified'}
Project root: rn-mri-clinical-fusion


In [2]:
manifest_path = ROOT / 'data/metadata/downloaded_imaging.jsonl'
if manifest_path.exists():
    manifest = pd.read_json(manifest_path, lines=True, dtype={'patient_id':str,'study_date_code':str})
    print('Downloaded DICOM files:', len(manifest))
    print('Patients:', manifest.patient_id.nunique())
    print('Patient-date pairs:', manifest[['patient_id','study_date_code']].drop_duplicates().shape[0])
    print('Uncompressed bytes:', int(manifest.size_bytes.sum()))
else:
    manifest = pd.DataFrame()
    print('Full selected-image download is not yet complete. Run the documented download script.')


Downloaded DICOM files: 40801
Patients: 59
Patient-date pairs: 107
Uncompressed bytes: 18208867252


In [3]:
import hashlib
if len(manifest):
    failures = []
    for record in manifest.to_dict('records'):
        p = ROOT / record['path']
        if not p.is_file() or p.stat().st_size != record['size_bytes']:
            failures.append(record['path'])
    assert not failures, failures[:5]
    sample = manifest.sample(n=min(20,len(manifest)), random_state=20261028)
    for record in sample.to_dict('records'):
        p = ROOT / record['path']
        assert hashlib.sha256(p.read_bytes()).hexdigest() == record['sha256']
    print('Every indexed file exists with the expected byte count; 20 deterministic SHA256 spot-checks passed.')


Every indexed file exists with the expected byte count; 20 deterministic SHA256 spot-checks passed.


In [4]:
series = pd.read_csv(ROOT / 'data/processed/image_series.csv')
print('Name-based sequence hints (not verified imaging labels):')
print(series[['flair_name_hint','t2_name_hint','t1_name_hint']].sum().to_string())
print('Image quality status: pending per-study review and target-lesion segmentation.')


Name-based sequence hints (not verified imaging labels):
flair_name_hint     637
t2_name_hint        826
t1_name_hint       1161
Image quality status: pending per-study review and target-lesion segmentation.
